# 05. Final Load Preparation — KPI Computation for Tableau
**Objective:** Compute all headline KPIs and export Tableau-ready summary tables.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/cleaned_retail_data.csv.gz', compression='gzip')
df_active = df[df['IsCancelled']==False].copy()
out_dir = '../data/processed/'
print(f"Loaded: {len(df):,} rows")

---
## 1. Headline KPIs

In [ ]:
total_revenue     = df_active['TotalRevenue'].sum()
total_orders      = df_active['InvoiceNo'].nunique()
total_customers   = df_active[df_active['CustomerID']!='Guest']['CustomerID'].nunique()
avg_order_value   = total_revenue / total_orders
total_items       = df_active['Quantity'].sum()
cancel_rate       = df['IsCancelled'].mean() * 100
repeat_customers  = df_active[df_active['IsRepeatCustomer']=='Repeat']['CustomerID'].nunique()
repeat_rate       = repeat_customers / total_customers * 100

print("=" * 50)
print("  HEADLINE KPIs — For Tableau Dashboard Tiles")
print("=" * 50)
print(f"  Total Revenue          : £{total_revenue:,.0f}")
print(f"  Total Orders           : {total_orders:,}")
print(f"  Total Unique Customers : {total_customers:,}")
print(f"  Avg Order Value        : £{avg_order_value:.2f}")
print(f"  Total Items Sold       : {total_items:,}")
print(f"  Cancellation Rate      : {cancel_rate:.2f}%")
print(f"  Repeat Customer Rate   : {repeat_rate:.1f}%")
print("=" * 50)

---
## 2. Monthly Revenue Table (for Tableau Line Chart)

In [ ]:
monthly_kpi = df_active.groupby(['Year','MonthNumber','MonthName','InvoiceYearMonth','Quarter']).agg(
    Revenue        = ('TotalRevenue','sum'),
    Orders         = ('InvoiceNo','nunique'),
    Customers      = ('CustomerID','nunique'),
    ItemsSold      = ('Quantity','sum'),
).reset_index().sort_values(['Year','MonthNumber'])

monthly_kpi['AvgOrderValue'] = (monthly_kpi['Revenue'] / monthly_kpi['Orders']).round(2)
monthly_kpi.to_csv(out_dir + 'tableau_monthly_kpi.csv', index=False)
print("Saved: tableau_monthly_kpi.csv")
monthly_kpi.head(13)

---
## 3. Country Revenue Table (for Tableau Map & Bar Chart)

In [ ]:
country_kpi = df_active.groupby(['Country','IsUK']).agg(
    Revenue   = ('TotalRevenue','sum'),
    Orders    = ('InvoiceNo','nunique'),
    Customers = ('CustomerID','nunique'),
).reset_index().sort_values('Revenue', ascending=False)

country_kpi['AvgOrderValue'] = (country_kpi['Revenue']/country_kpi['Orders']).round(2)
country_kpi.to_csv(out_dir + 'tableau_country_kpi.csv', index=False)
print("Saved: tableau_country_kpi.csv")
country_kpi.head(10)

---
## 4. Product Revenue Table (for Tableau Bar Chart)

In [ ]:
product_kpi = df_active.groupby(['StockCode','Description']).agg(
    Revenue      = ('TotalRevenue','sum'),
    QuantitySold = ('Quantity','sum'),
    Orders       = ('InvoiceNo','nunique'),
).reset_index().sort_values('Revenue', ascending=False)

product_kpi.to_csv(out_dir + 'tableau_product_kpi.csv', index=False)
print("Saved: tableau_product_kpi.csv")
product_kpi.head(10)

---
## 5. Customer Segment Summary (for Tableau Segment View)

In [ ]:
segment_kpi = df_active[df_active['CustomerID']!='Guest'].drop_duplicates('CustomerID').groupby('CustomerSegment').agg(
    Customers = ('CustomerID','nunique'),
).reset_index()
segment_kpi['Percentage'] = (segment_kpi['Customers']/segment_kpi['Customers'].sum()*100).round(1)
segment_kpi = segment_kpi.sort_values('Customers', ascending=False)

segment_kpi.to_csv(out_dir + 'tableau_segment_kpi.csv', index=False)
print("Saved: tableau_segment_kpi.csv")
segment_kpi

---
## 6. Time Analysis Table (for Tableau Heatmap)

In [ ]:
time_kpi = df_active.groupby(['DayOfWeek','TimeOfDay','Hour']).agg(
    Revenue = ('TotalRevenue','sum'),
    Orders  = ('InvoiceNo','nunique'),
).reset_index()

time_kpi.to_csv(out_dir + 'tableau_time_kpi.csv', index=False)
print("Saved: tableau_time_kpi.csv")
time_kpi.head(10)

---
## 7. Basket Size Summary

In [ ]:
basket_kpi = df_active.drop_duplicates('InvoiceNo').groupby('BasketSize').agg(
    Orders  = ('InvoiceNo','count'),
    Revenue = ('TotalRevenue','sum'),
).reset_index()
basket_kpi['AvgOrderValue'] = (basket_kpi['Revenue']/basket_kpi['Orders']).round(2)

basket_kpi.to_csv(out_dir + 'tableau_basket_kpi.csv', index=False)
print("Saved: tableau_basket_kpi.csv")
basket_kpi

---
## 8. Final Summary

In [ ]:
print("=" * 60)
print("  FINAL LOAD PREP COMPLETE — All Tableau CSV files saved")
print("=" * 60)
import os
files = [f for f in os.listdir(out_dir) if f.startswith('tableau_')]
for f in files:
    size = os.path.getsize(out_dir+f)
    print(f"  {f:<40} ({size/1024:.1f} KB)")
print("\n  Import cleaned_retail_data.csv.gz as the main Tableau source.")
print("  Use individual tableau_*.csv files for pre-aggregated views.")
print("=" * 60)